# Import

In [1]:
import time
import nidaqmx
import HEDS
from hedslib.heds_types import *
#from instrcomms import Communications
from pylablib.devices.Thorlabs import MFF
from pylablib.devices.Thorlabs import PM160

In [2]:
import os
import numpy.random as rd
import matplotlib.pyplot as plt
from functools import partial
import numpy as np
import pyvisa as pvisa
import qcodes as qc
from qcodes.instrument_drivers.yehlab_driver.QD import QD
import qcodes.instrument_drivers.nplab_drivers as npd
import qcodes.instrument_drivers.yehlab_driver as yd
from qcodes.instrument_drivers.stanford_research.SR830 import SR830
from qcodes.instrument_drivers.Keithley.Keithley_2636B import *

# Check address (if the connection failed)

In [64]:
rm = pvisa.ResourceManager()
resources = rm.list_resources()
print("List")
for res in resources:
    print(res)

List
GPIB0::13::INSTR
GPIB0::2::INSTR
GPIB1::3::INSTR
GPIB1::8::INSTR
PXI0::4-0.0::INSTR
USB0::0x05E6::0x2636::4353546::0::INSTR
USB0::0x05E6::0x2636::4437673::0::INSTR
USB0::0x0957::0x3D18::MY59030016::0::INSTR
USB0::0x1313::0x8078::P0005244::0::INSTR
USB0::0x1313::0x8078::PM001388::0::INSTR
USB0::0x2EC7::0x6522::802557043747210033::0::INSTR
ASRL3::INSTR
ASRL5::INSTR


# Power setting connection and test (Flip mount/Power meter/ND filter)

In [ ]:
# Warning! Before setting the power, rotate the second ND filter to the clear part to prevent damage the SLM during the operation.

# --- Flip mount of the power meter ---

In [3]:
#Import pyft232
serial = "37000705"
flip_mount = MFF(serial)
flip_mount.open()  # explicitly open the FTDI backend
flip_mount.move_to_state(0) #Block

C:\Users\HP\.conda\envs\LL_Group_qcodes\lib\site-packages\pylablib\devices\Thorlabs\kinesis.py:230: UserWarning: model number MFF002 doesn't match the device ID prefix 37(MFF10.)
  warnings.warn("model number {} doesn't match the device ID prefix {}({})".format(model_no,port,port_model_no))


In [ ]:
# Flip mount control
flip_mount.move_to_state(0) #Block

# --- Power meter ---

In [4]:
pm = PM160('USB0::0x1313::0x8078::P0005244::0::INSTR')
pm.open()
pm.enable_autorange
# Check settings
print("Power meter status: ", pm.get_full_status(include="all"))

{'mode': 'power',
 'autorange': True,
 'wavelength': 6.33e-07,
 'range': 0.0001477929,
 'reading': 1.80103634e-05}

In [5]:
# Change wavelength in meter
pm.set_wavelength(633e-09)

6.33e-07

In [6]:
# Read power (in Watts)
power = pm.get_power()
print(f"Measured power: {power} W")

Measured power: 1.09201119e-05 W


# --- ND filter motor ---

In [7]:
DEV = "Dev3"
pul_task = nidaqmx.Task()
dir_task = nidaqmx.Task()
ena_task = nidaqmx.Task()

pul_task.do_channels.add_do_chan(f"{DEV}/port2/line0")  # PUL
dir_task.do_channels.add_do_chan(f"{DEV}/port2/line2")  # DIR
ena_task.do_channels.add_do_chan(f"{DEV}/port2/line1")  # ENA

delay = 0.00        # step pulse delay

def step_motor(steps, direction):
    dir_task.write(direction)
    for _ in range(steps):
        pul_task.write(True)
        time.sleep(delay)
        pul_task.write(False)
        time.sleep(delay)


# Set power

In [14]:
def set_power(setpoint, tolerance):
    flip_mount.move_to_state(0) #Block

    # Enable motor
    ena_task.write(False)

    # Feedback loop
    ena_task.write(False)
    for i in range(2000):  # max iterations (safety), 0.36 deg/step
        power = pm.get_power()
        error = setpoint - power
        #print(f"Measured: {power:.6f} W, Error: {error:.6e} W")

        if abs(error) <= tolerance:
            ena_task.write(True)
            print("Target reached.")
            time.sleep(5) # Wait till the motor completely stop
            power = pm.get_power()
            error = setpoint - power
            print(f"Measured: {power:.6f} W, Error: {error:.6e} W")

            break

        if error > 0:   # measured too low → need more power → rotate back
            step_motor(1, True)
        else:           # measured too high → need less power → rotate forward
            step_motor(1, False)

    # Disable motor
    ena_task.write(True)
    #pul_task.close()
    #dir_task.close()
    #ena_task.close()
    #pm.close()
    time.sleep(2)
    flip_mount.move_to_state(1) #Remove

In [17]:
setpoint = 90e-6     # target power in W
tolerance = 1e-6     # acceptable error
set_power(setpoint, tolerance)

Measured: 0.000000 W, Error: 7.979711e-05 W
Measured: 0.000000 W, Error: 7.959983e-05 W
Measured: 0.000000 W, Error: 7.974894e-05 W
Measured: 0.000000 W, Error: 7.974037e-05 W
Measured: 0.000000 W, Error: 7.971958e-05 W
Measured: 0.000000 W, Error: 7.982680e-05 W
Measured: 0.000000 W, Error: 7.986111e-05 W
Measured: 0.000000 W, Error: 7.982020e-05 W
Measured: 0.000000 W, Error: 7.982416e-05 W
Measured: 0.000000 W, Error: 7.979744e-05 W
Measured: 0.000000 W, Error: 7.985187e-05 W
Measured: 0.000000 W, Error: 7.982020e-05 W
Measured: 0.000000 W, Error: 7.990796e-05 W
Measured: 0.000000 W, Error: 7.985979e-05 W
Measured: 0.000000 W, Error: 7.974960e-05 W
Measured: 0.000000 W, Error: 7.986969e-05 W
Measured: 0.000000 W, Error: 7.990895e-05 W
Measured: 0.000000 W, Error: 7.986210e-05 W
Measured: 0.000000 W, Error: 7.985583e-05 W
Measured: 0.000000 W, Error: 7.976610e-05 W
Measured: 0.000001 W, Error: 7.939166e-05 W
Measured: 0.000002 W, Error: 7.805622e-05 W
Measured: 0.000002 W, Error: 7.7

Measured: 0.000004 W, Error: 7.550542e-05 W
Measured: 0.000005 W, Error: 7.538401e-05 W
Measured: 0.000005 W, Error: 7.542162e-05 W
Measured: 0.000005 W, Error: 7.535762e-05 W
Measured: 0.000005 W, Error: 7.533585e-05 W
Measured: 0.000005 W, Error: 7.517420e-05 W
Measured: 0.000005 W, Error: 7.525008e-05 W
Measured: 0.000005 W, Error: 7.510195e-05 W
Measured: 0.000005 W, Error: 7.499077e-05 W
Measured: 0.000005 W, Error: 7.501947e-05 W
Measured: 0.000005 W, Error: 7.494657e-05 W
Measured: 0.000005 W, Error: 7.482549e-05 W
Measured: 0.000005 W, Error: 7.487036e-05 W
Measured: 0.000005 W, Error: 7.482186e-05 W
Measured: 0.000005 W, Error: 7.474599e-05 W
Measured: 0.000005 W, Error: 7.475357e-05 W
Measured: 0.000005 W, Error: 7.469386e-05 W
Measured: 0.000005 W, Error: 7.469716e-05 W
Measured: 0.000005 W, Error: 7.459819e-05 W
Measured: 0.000005 W, Error: 7.466879e-05 W
Measured: 0.000005 W, Error: 7.452990e-05 W
Measured: 0.000006 W, Error: 7.448932e-05 W
Measured: 0.000005 W, Error: 7.4

Measured: 0.000000 W, Error: 7.987398e-05 W
Measured: 0.000000 W, Error: 7.982746e-05 W
Measured: 0.000000 W, Error: 7.978886e-05 W
Measured: 0.000000 W, Error: 7.977138e-05 W
Measured: 0.000000 W, Error: 7.970705e-05 W
Measured: 0.000000 W, Error: 7.974828e-05 W
Measured: 0.000000 W, Error: 7.984198e-05 W
Measured: 0.000000 W, Error: 7.979381e-05 W
Measured: 0.000000 W, Error: 7.977369e-05 W
Measured: 0.000000 W, Error: 7.981492e-05 W
Measured: 0.000000 W, Error: 7.981163e-05 W
Measured: 0.000000 W, Error: 7.984396e-05 W
Measured: 0.000000 W, Error: 7.973707e-05 W
Measured: 0.000000 W, Error: 7.982383e-05 W
Measured: 0.000000 W, Error: 7.982944e-05 W
Measured: 0.000000 W, Error: 7.971760e-05 W
Measured: 0.000000 W, Error: 7.982152e-05 W
Measured: 0.000000 W, Error: 7.971760e-05 W
Measured: 0.000000 W, Error: 7.983175e-05 W
Measured: 0.000000 W, Error: 7.967802e-05 W
Measured: 0.000000 W, Error: 7.986474e-05 W
Measured: 0.000000 W, Error: 7.987497e-05 W
Measured: 0.000000 W, Error: 7.9

Measured: 0.000039 W, Error: 4.138163e-05 W
Measured: 0.000039 W, Error: 4.090206e-05 W
Measured: 0.000039 W, Error: 4.094202e-05 W
Measured: 0.000040 W, Error: 3.999621e-05 W
Measured: 0.000040 W, Error: 4.005949e-05 W
Measured: 0.000041 W, Error: 3.901710e-05 W
Measured: 0.000041 W, Error: 3.941007e-05 W
Measured: 0.000041 W, Error: 3.944671e-05 W
Measured: 0.000041 W, Error: 3.855751e-05 W
Measured: 0.000042 W, Error: 3.838767e-05 W
Measured: 0.000041 W, Error: 3.863411e-05 W
Measured: 0.000043 W, Error: 3.746184e-05 W
Measured: 0.000043 W, Error: 3.727534e-05 W
Measured: 0.000043 W, Error: 3.652602e-05 W
Measured: 0.000043 W, Error: 3.698227e-05 W
Measured: 0.000043 W, Error: 3.663591e-05 W
Measured: 0.000044 W, Error: 3.595320e-05 W
Measured: 0.000044 W, Error: 3.604978e-05 W
Measured: 0.000045 W, Error: 3.524052e-05 W
Measured: 0.000045 W, Error: 3.474763e-05 W
Measured: 0.000046 W, Error: 3.353206e-05 W
Measured: 0.000046 W, Error: 3.403494e-05 W
Measured: 0.000047 W, Error: 3.2

Measured: 0.000054 W, Error: 2.551599e-05 W
Measured: 0.000054 W, Error: 2.568916e-05 W
Measured: 0.000055 W, Error: 2.537944e-05 W
Measured: 0.000055 W, Error: 2.478331e-05 W
Measured: 0.000055 W, Error: 2.520293e-05 W
Measured: 0.000054 W, Error: 2.576575e-05 W
Measured: 0.000054 W, Error: 2.567917e-05 W
Measured: 0.000055 W, Error: 2.541940e-05 W
Measured: 0.000054 W, Error: 2.592228e-05 W
Measured: 0.000054 W, Error: 2.610545e-05 W
Measured: 0.000054 W, Error: 2.567917e-05 W
Measured: 0.000055 W, Error: 2.491652e-05 W
Measured: 0.000055 W, Error: 2.545937e-05 W
Measured: 0.000054 W, Error: 2.569582e-05 W
Measured: 0.000054 W, Error: 2.571247e-05 W
Measured: 0.000054 W, Error: 2.553929e-05 W
Measured: 0.000054 W, Error: 2.593893e-05 W
Measured: 0.000055 W, Error: 2.537278e-05 W
Measured: 0.000054 W, Error: 2.587233e-05 W
Measured: 0.000055 W, Error: 2.544605e-05 W
Measured: 0.000055 W, Error: 2.540609e-05 W
Measured: 0.000055 W, Error: 2.541940e-05 W
Measured: 0.000054 W, Error: 2.5

...

Measured: 0.000000 W, Error: 7.986969e-05 W
Measured: 0.000000 W, Error: 7.991159e-05 W
Measured: 0.000000 W, Error: 7.991291e-05 W
Measured: 0.000000 W, Error: 7.993039e-05 W
Measured: 0.000000 W, Error: 7.993138e-05 W
Measured: 0.000000 W, Error: 7.990697e-05 W
Measured: 0.000000 W, Error: 7.986969e-05 W
Measured: 0.000000 W, Error: 7.987563e-05 W
Measured: 0.000000 W, Error: 7.994095e-05 W
Measured: 0.000000 W, Error: 7.993501e-05 W
Measured: 0.000000 W, Error: 7.983043e-05 W
Measured: 0.000000 W, Error: 7.989905e-05 W
Measured: 0.000000 W, Error: 7.986309e-05 W
Measured: 0.000000 W, Error: 7.996833e-05 W
Measured: 0.000000 W, Error: 7.983538e-05 W
Measured: 0.000000 W, Error: 7.995414e-05 W
Measured: 0.000000 W, Error: 7.986540e-05 W
Measured: 0.000000 W, Error: 7.991653e-05 W
Measured: 0.000000 W, Error: 7.987167e-05 W
Measured: 0.000000 W, Error: 7.986870e-05 W
Measured: 0.000000 W, Error: 7.992610e-05 W
Measured: 0.000000 W, Error: 7.988717e-05 W
Measured: 0.000000 W, Error: 7.9

Measured: 0.000000 W, Error: 7.989575e-05 W
Measured: 0.000000 W, Error: 7.994524e-05 W
Measured: 0.000000 W, Error: 7.983604e-05 W
Measured: 0.000000 W, Error: 7.992115e-05 W
Measured: 0.000000 W, Error: 7.986045e-05 W
Measured: 0.000000 W, Error: 7.995909e-05 W
Measured: 0.000000 W, Error: 7.993435e-05 W
Measured: 0.000000 W, Error: 7.986342e-05 W
Measured: 0.000000 W, Error: 7.987596e-05 W
Measured: 0.000000 W, Error: 7.998086e-05 W
Measured: 0.000000 W, Error: 7.985913e-05 W
Measured: 0.000000 W, Error: 7.991554e-05 W
Measured: 0.000000 W, Error: 7.985220e-05 W
Measured: 0.000000 W, Error: 7.993864e-05 W
Measured: 0.000000 W, Error: 7.994557e-05 W
Measured: 0.000000 W, Error: 7.983010e-05 W
Measured: 0.000000 W, Error: 7.990928e-05 W
Measured: 0.000000 W, Error: 7.994392e-05 W
Measured: 0.000000 W, Error: 7.989509e-05 W
Measured: 0.000000 W, Error: 7.985484e-05 W
Measured: 0.000000 W, Error: 7.995876e-05 W
Measured: 0.000000 W, Error: 7.984198e-05 W
Measured: 0.000000 W, Error: 7.9

Measured: 0.000000 W, Error: 7.990235e-05 W
Measured: 0.000000 W, Error: 7.990301e-05 W
Measured: 0.000000 W, Error: 7.985253e-05 W
Measured: 0.000000 W, Error: 7.988189e-05 W
Measured: 0.000000 W, Error: 7.986540e-05 W
Measured: 0.000000 W, Error: 7.996305e-05 W
Measured: 0.000000 W, Error: 7.983835e-05 W
Measured: 0.000000 W, Error: 7.995480e-05 W
Measured: 0.000000 W, Error: 7.989839e-05 W
Measured: 0.000000 W, Error: 7.987893e-05 W
Measured: 0.000000 W, Error: 7.992181e-05 W
Measured: 0.000000 W, Error: 7.988288e-05 W
Measured: 0.000000 W, Error: 7.989344e-05 W
Measured: 0.000000 W, Error: 7.986441e-05 W
Measured: 0.000000 W, Error: 7.990697e-05 W
Measured: 0.000000 W, Error: 7.991126e-05 W
Measured: 0.000000 W, Error: 7.986375e-05 W
Measured: 0.000000 W, Error: 7.993369e-05 W
Measured: 0.000000 W, Error: 7.985319e-05 W
Measured: 0.000000 W, Error: 7.989047e-05 W
Measured: 0.000000 W, Error: 7.989938e-05 W
Measured: 0.000000 W, Error: 7.984033e-05 W
Measured: 0.000000 W, Error: 7.9

Measured: 0.000000 W, Error: 7.987464e-05 W
Measured: 0.000000 W, Error: 7.984528e-05 W
Measured: 0.000000 W, Error: 7.993633e-05 W
Measured: 0.000000 W, Error: 7.994062e-05 W
Measured: 0.000000 W, Error: 7.988222e-05 W
Measured: 0.000000 W, Error: 7.991159e-05 W
Measured: 0.000000 W, Error: 7.995876e-05 W
Measured: 0.000000 W, Error: 7.992280e-05 W
Measured: 0.000000 W, Error: 7.991455e-05 W
Measured: 0.000000 W, Error: 7.987398e-05 W
Measured: 0.000000 W, Error: 7.994656e-05 W
Measured: 0.000000 W, Error: 7.983934e-05 W
Measured: 0.000000 W, Error: 7.989971e-05 W
Measured: 0.000000 W, Error: 7.993666e-05 W
Measured: 0.000000 W, Error: 7.990961e-05 W
Measured: 0.000000 W, Error: 7.992742e-05 W
Measured: 0.000000 W, Error: 7.990400e-05 W
Measured: 0.000000 W, Error: 7.988321e-05 W
Measured: 0.000000 W, Error: 7.989443e-05 W
Measured: 0.000000 W, Error: 7.993435e-05 W
Measured: 0.000000 W, Error: 7.983835e-05 W
Measured: 0.000000 W, Error: 7.989377e-05 W
Measured: 0.000000 W, Error: 7.9

Measured: 0.000000 W, Error: 7.987926e-05 W
Measured: 0.000000 W, Error: 7.984627e-05 W
Measured: 0.000000 W, Error: 7.994128e-05 W
Measured: 0.000000 W, Error: 7.991225e-05 W
Measured: 0.000000 W, Error: 7.989608e-05 W
Measured: 0.000000 W, Error: 7.984363e-05 W
Measured: 0.000000 W, Error: 7.994721e-05 W
Measured: 0.000000 W, Error: 7.986606e-05 W
Measured: 0.000000 W, Error: 7.994820e-05 W
Measured: 0.000000 W, Error: 7.991192e-05 W
Measured: 0.000000 W, Error: 7.985550e-05 W
Measured: 0.000000 W, Error: 7.991851e-05 W
Measured: 0.000000 W, Error: 7.989674e-05 W
Measured: 0.000000 W, Error: 7.988057e-05 W
Measured: 0.000000 W, Error: 7.989245e-05 W
Measured: 0.000000 W, Error: 7.990235e-05 W
Measured: 0.000000 W, Error: 7.991587e-05 W
Measured: 0.000000 W, Error: 7.987563e-05 W
Measured: 0.000000 W, Error: 7.989377e-05 W
Measured: 0.000000 W, Error: 7.990169e-05 W
Measured: 0.000000 W, Error: 7.993567e-05 W
Measured: 0.000000 W, Error: 7.991225e-05 W
Measured: 0.000000 W, Error: 7.9

# SLM

In [18]:
# Init HOLOEYE SLM Display SDK and make sure to check for the correct version this script was written with:
err = HEDS.SDK.Init(4,0)
assert err == HEDSERR_NoError, HEDS.SDK.ErrorString(err)

# Open device detection and retrieve one SLM, and open the SLM Preview window in "Fit" mode for the selected SLM:
slm = HEDS.SLM.Init("", True, 0.0)
assert slm.errorCode() == HEDSERR_NoError, HEDS.SDK.ErrorString(slm.errorCode())

# Move SLM preview on a secondary monitor, if available, to enlarge it: 
slm.preview().autoplaceLayoutOnSecondaryMonitor()

0

In [12]:
# Vortex phase definition
def set_OAM(l, shift_x = 300, shift_y = 0):
    # Define your desired shift (in pixels, relative to the SLM center)
    #shift_x = 310   # positive = right
    #shift_y = 30  # positive = down

    # Show a vortex shifted from center

    err = slm.showVortex(
        l,
        centerX = shift_x,
        centerY = shift_y
    )


In [24]:
# Set OAM and offset
set_OAM(5, shift_x = 300, shift_y = 0)

# Measurement loop

In [26]:
OAM = [-3, -2, -1, 0, 1, 2, 3]
Power = [1.72e-3, 1.74e-3, 1.75e-3, 1.76e-3, 1.75e-3, 1.73e-3] #Power for each OAM
tolerance = 1e-6 
#cycle_number = 40


for i in range(len(OAM)):
    set_OAM(OAM[i])
    setpoint = Power[i]
    set_power(setpoint, 1e-6)
    #lOCK MEASUREMENT
    
    time.sleep(1)
    flip_mount.move_to_state(0) #Block
    
   

Measured: 0.001152 W, Error: 5.677979e-04 W
Measured: 0.001145 W, Error: 5.751330e-04 W
Measured: 0.001142 W, Error: 5.783337e-04 W
Measured: 0.001141 W, Error: 5.791672e-04 W
Measured: 0.001140 W, Error: 5.796340e-04 W
Measured: 0.001146 W, Error: 5.741327e-04 W
Measured: 0.001154 W, Error: 5.659642e-04 W
Measured: 0.001145 W, Error: 5.746995e-04 W
Measured: 0.001164 W, Error: 5.558618e-04 W
Measured: 0.001177 W, Error: 5.431922e-04 W
Measured: 0.001183 W, Error: 5.367240e-04 W
Measured: 0.001191 W, Error: 5.288555e-04 W
Measured: 0.001227 W, Error: 4.930472e-04 W
Measured: 0.001214 W, Error: 5.061169e-04 W
Measured: 0.001255 W, Error: 4.646072e-04 W
Measured: 0.001269 W, Error: 4.508041e-04 W
Measured: 0.001287 W, Error: 4.328665e-04 W
Measured: 0.001299 W, Error: 4.209971e-04 W
Measured: 0.001309 W, Error: 4.107280e-04 W
Measured: 0.001321 W, Error: 3.989586e-04 W
Measured: 0.001329 W, Error: 3.911900e-04 W
Measured: 0.001330 W, Error: 3.897898e-04 W
Measured: 0.001357 W, Error: 3.6

Measured: 0.001725 W, Error: 2.536475e-05 W
Measured: 0.001708 W, Error: 4.246348e-05 W
Measured: 0.001763 W, Error: -1.252134e-05 W
Measured: 0.001767 W, Error: -1.721509e-05 W
Measured: 0.001741 W, Error: 8.935990e-06 W
Measured: 0.001755 W, Error: -5.145030e-06 W
Measured: 0.001785 W, Error: -3.464921e-05 W
Measured: 0.001763 W, Error: -1.319189e-05 W
Measured: 0.001671 W, Error: 7.900777e-05 W
Measured: 0.001736 W, Error: 1.362985e-05 W
Measured: 0.001715 W, Error: 3.508717e-05 W
Measured: 0.001803 W, Error: -5.275354e-05 W
Measured: 0.001745 W, Error: 5.248070e-06 W
Measured: 0.001750 W, Error: -1.163500e-07 W
Target reached.
Measured: 0.001743 W, Error: 7.259950e-06 W
Measured: 0.000053 W, Error: 1.707308e-03 W
Measured: 0.000037 W, Error: 1.722710e-03 W
Measured: 0.000026 W, Error: 1.734267e-03 W
Measured: 0.000041 W, Error: 1.718994e-03 W
Measured: 0.000035 W, Error: 1.724962e-03 W
Measured: 0.000032 W, Error: 1.728235e-03 W
Measured: 0.000024 W, Error: 1.735682e-03 W
Measured:

Measured: 0.001714 W, Error: 1.609312e-05 W
Measured: 0.001755 W, Error: -2.548031e-05 W
Measured: 0.001725 W, Error: 5.029360e-06 W
Measured: 0.001738 W, Error: -8.046180e-06 W
Measured: 0.001723 W, Error: 6.705390e-06 W
Measured: 0.001704 W, Error: 2.615140e-05 W
Measured: 0.001710 W, Error: 1.978104e-05 W
Measured: 0.001739 W, Error: -8.716850e-06 W
Measured: 0.001723 W, Error: 7.376060e-06 W
Measured: 0.001792 W, Error: -6.236033e-05 W
Measured: 0.001709 W, Error: 2.078699e-05 W
Measured: 0.001747 W, Error: -1.709852e-05 W
Measured: 0.001715 W, Error: 1.542245e-05 W
Measured: 0.001715 W, Error: 1.542245e-05 W
Measured: 0.001729 W, Error: 6.708800e-07 W
Target reached.
Measured: 0.001721 W, Error: 8.717280e-06 W


IndexError: list index out of range

# 2636B

In [9]:
k2636B = Keithley2600('k2636B','USB0::0x05E6::0x2636::4437673::0::INSTR')

Connected to: Keithley Instruments Inc. 2636B (serial:4437673, firmware:3.3.5) in 0.04s


In [21]:
smua_mode = k2636B.smua.mode()
print(f"SMUA Mode: {smua_mode}")
smub_mode = k2636B.smub.mode()
print(f"SMUB Mode: {smub_mode}")

SMUA Mode: voltage
SMUB Mode: voltage


In [29]:
#Define the input values (Vd)
startVd = -3
stopVd = 3
stepVd = 0.2
vol_val = np.arange(startVd, stopVd + stepVd, stepVd)
if startVd * stepVd < 0:
    point0 = int(startVd / stepVd)
    vol_val[point0 - 1]=0
print(vol_val)

[-3.  -2.8 -2.6 -2.4 -2.2 -2.  -1.8 -1.6 -1.4 -1.2 -1.  -0.8 -0.6 -0.4
 -0.2  0.   0.2  0.4  0.6  0.8  1.   1.2  1.4  1.6  1.8  2.   2.2  2.4
  2.6  2.8  3. ]


In [32]:
#Dual sweep
vol_val = np.append(np.flip(vol_val), vol_val)
print(vol_val)

[ 3.   2.8  2.6  2.4  2.2  2.   1.8  1.6  1.4  1.2  1.   0.8  0.6  0.4
  0.2  0.  -0.2 -0.4 -0.6 -0.8 -1.  -1.2 -1.4 -1.6 -1.8 -2.  -2.2 -2.4
 -2.6 -2.8 -3.  -3.  -2.8 -2.6 -2.4 -2.2 -2.  -1.8 -1.6 -1.4 -1.2 -1.
 -0.8 -0.6 -0.4 -0.2  0.   0.2  0.4  0.6  0.8  1.   1.2  1.4  1.6  1.8
  2.   2.2  2.4  2.6  2.8  3. ]


In [ ]:
"""
my4200.query("BC") # Clears all readings from the buffer
my4200.query("DE") # Channel definition page
my4200.query("CH1, 'VD', 'ID', 1, 2") 
my4200.query("CH2, 'VG', 'IG', 1, 1")
#my4200.query("CH3, 'VB', 'IB', 1, 2")
my4200.query("SS") # Source setup page
my4200.query("VR2, -5, 5, 0.1, 10e-6")
my4200.query("VP1, -10, 4, 10e-6")
#my4200.query("VC1, 0, 100e-3")
my4200.query("HT 0") # Hold time, delays the start of a sweep
my4200.query("DT 0") # Delay time, wait between the output voltage is set
my4200.query("IT2") # Integration time
my4200.query("RS 5") # Resolution in number of digits
my4200.query("RG 1, 1e-9") #Lowest range
my4200.query("RG 2, 1e-9")
#my4200.query("RG 3, 100e-9")
my4200.query("SM") # Measurment setup page
my4200.query("DM1") # Display modes
my4200.query("XN 'VG', 1, -5, 5") # X-axis
my4200.query("YA 'ID', 1, -1e-4, 1e-4") # Y-axis1
#my4200.query("YB 'ID', 1, -1e-4, 1e-4") # Y-axis2
"""
"""
    for i in range(2000):  # max iterations (safety), 0.36 deg/step
    power = pm.get_power()
    error = setpoint - power
    print(f"Measured: {power:.6f} W, Error: {error:.6e} W")

    if abs(error) <= tolerance:
        print("Target reached.")
        break

        if error > 0:   # measured too low → need more power → rotate back
            step_motor(10, True)
        else:           # measured too high → need less power → rotate forward
            step_motor(10, False)
    ena_task.write(True)
    flip_mount.move_to_state(1) #Remove
"""
"""
    #Potentiation
    
    my4200.query("BC") # Clears all readings from the buffer
    my4200.query("DE") # Channel definition page
    my4200.query("CH1, 'VD', 'ID', 1, 2") 
    my4200.query("CH2, 'VG', 'IG', 1, 1")
    #my4200.query("CH3, 'VB', 'IB', 1, 2")
    my4200.query("SS") # Source setup page
    my4200.query("VR2, -5, 5, 0.1, 10e-6")
    my4200.query("VP1, -10, 4, 10e-6")
    #my4200.query("VC1, 0, 100e-3")
    my4200.query("HT 0") # Hold time, delays the start of a sweep
    my4200.query("DT 0") # Delay time, wait between the output voltage is set
    my4200.query("IT2") # Integration time
    my4200.query("RS 5") # Resolution in number of digits
    my4200.query("RG 1, 1e-9") #Lowest range
    my4200.query("RG 2, 1e-9")
    #my4200.query("RG 3, 100e-9")
    my4200.query("SM") # Measurment setup page
    my4200.query("DM1") # Display modes
    my4200.query("XN 'VG', 1, -5, 5") # X-axis
    my4200.query("YA 'ID', 1, -1e-4, 1e-4") # Y-axis1
    #my4200.query("YB 'ID', 1, -1e-4, 1e-4") # Y-axis2    
    
    for i in range(cycle_number):
        #Run IVg
        #my4200.query("BC")     # Clear or not?
        #my4200.query("MD") # Measurement control page
        my4200.query("ME3") # Control measurment
"""
"""
        ME
        Measurement action:
        ▪ Run a single trigger test and store readings in a cleared buffer: 1
        ▪ Run a repeat trigger test and store readings in a cleared buffer: 2
        ▪ Run a single trigger test and append readings to the buffer: 3
        ▪ Abort the test: 4
"""
"""
        # wait for measurement to complete
        status = my4200.query("SP")
        
        while int(status) != 1:
            status = my4200.query("SP")
            time.sleep(1)
        
    # Save data
    my4200.query("SV 'D FileName Comment'")
    
    #Depression

    my4200.query("BC") # Clears all readings from the buffer
    my4200.query("DE") # Channel definition page
    my4200.query("CH1, 'VD', 'ID', 1, 2") 
    my4200.query("CH2, 'VG', 'IG', 1, 1")
    #my4200.query("CH3, 'VB', 'IB', 1, 2")
    my4200.query("SS") # Source setup page
    my4200.query("VR2, -5, 5, 0.1, 10e-6")
    my4200.query("VP1, -10, 4, 10e-6")
    #my4200.query("VC1, 0, 100e-3")
    my4200.query("HT 0") # Hold time, delays the start of a sweep
    my4200.query("DT 0") # Delay time, wait between the output voltage is set
    my4200.query("IT2") # Integration time
    my4200.query("RS 5") # Resolution in number of digits
    my4200.query("RG 1, 1e-9") #Lowest range
    my4200.query("RG 2, 1e-9")
    #my4200.query("RG 3, 100e-9")
    my4200.query("SM") # Measurment setup page
    my4200.query("DM1") # Display modes
    my4200.query("XN 'VG', 1, -5, 5") # X-axis
    my4200.query("YA 'ID', 1, -1e-4, 1e-4") # Y-axis1
    #my4200.query("YB 'ID', 1, -1e-4, 1e-4") # Y-axis2
    
    for i in range(cycle_number):
        #Run IVg
        #my4200.query("BC")     # Clear or not?
        #my4200.query("MD") # Measurement control page
        my4200.query("ME3") # Control measurment
"""
"""
        ME
        Measurement action:
        ▪ Run a single trigger test and store readings in a cleared buffer: 1
        ▪ Run a repeat trigger test and store readings in a cleared buffer: 2
        ▪ Run a single trigger test and append readings to the buffer: 3
        ▪ Abort the test: 4
"""
"""        
        # wait for measurement to complete
        status = my4200.query("SP")
        
        while int(status) != 1:
            status = my4200.query("SP")
            time.sleep(1)
    
    # Save data
    my4200.query("SV 'D FileName Comment'")
"""     

In [ ]:
"""
# Device name (check NI MAX, usually "Dev3")
DEV = "Dev3"

# We want lines 0, 1, 2 on port2
LINES = f"{DEV}/port2/line0:2"  # shorthand for line0 through line2

with nidaqmx.Task() as task:
    task.do_channels.add_do_chan(LINES)

    # ---- Initial state ----
    task.write([False, False, False], auto_start=True)
    # PUL=False, ENA=False (enabled), DIR=False

    print("Motor enabled, direction set.")

    steps = 20000
    delay = 0.0

    for i in range(steps):
        # Pulse HIGH
        task.write([False, False, True], auto_start=True)  
        time.sleep(delay)
        # Pulse LOW
        task.write([False, False, False], auto_start=True)
        time.sleep(delay)

    print("Done stepping.")

    # Disable motor
    task.write([False, True, False], auto_start=True)  # ENA=HIGH (disable)
    print("Motor disabled.")
"""

In [3]:
# Functions of 4200

import time
from instrcomms import Communications

class Keithley4200:
    def __init__(self, resource_str):
        self.conn = Communications(resource_str)

    def connect(self):
        self.conn.connect()
        # Ethernet (KXCI socket) requires NULL terminator
        self.conn._instrument_object.write_termination = "\0"
        self.conn._instrument_object.read_termination = "\0"

    def disconnect(self):
        self.conn.disconnect()

    def send(self, cmd, expect_response=False):
        """Send a command, return response if requested."""
        if expect_response:
            return self.conn.query(cmd)
        else:
            self.conn.write(cmd)

    def wait_for_completion(self, poll_interval=1):
        """Poll until measurement is finished (SP=1)."""
        status = self.send("SP", expect_response=True)
        while int(status) != 1:
            time.sleep(poll_interval)
            status = self.send("SP", expect_response=True)

    def save_data(self, filename="D_FileName", comment="Comment"):
        self.send(f"SV '{filename} {comment}'")

    # -----------------------------
    # Measurement setups
    # -----------------------------
    def run_IVD(self, VD_start, VD_stop, VD_step, VG_start, VG_step, VG_step_num, 
                  ID_compliance = 10e-6, IG_compliance = 10e-6, HT = 0, DT = 0, IT = 2, 
                  RS = 5, ID_lowrange = 1e-9, IG_lowrange = 1e-9):
        """ID–VD measurement, sweeping Vd with fixed Vg."""
        self.send("BC")  # clear buffers
        self.send("DE")  # channel definition page
        self.send("CH1, 'VD', 'ID', 1, 1") 
        self.send("CH2, 'VG', 'IG', 1, 2")

        self.send("SS")  # source setup page
        self.send(f"VR1, {VD_start}, {VD_stop}, {VD_step}, {ID_compliance}")   
        self.send(f"VP2, {VG_start}, {VG_step}, {IG_compliance}")       
        self.send(f"HT {HT}")
        self.send(f"DT {DT}")
        self.send(f"IT{IT}")
        self.send(f"RS {RS}")
        self.send(f"RG 1, {ID_lowrange}")
        self.send(f"RG 2, {IG_lowrange}")

        self.send("SM")   # measurement setup
        self.send("DM1")  # display mode
        self.send(f"XN 'VD', 1, {VD_start}, {VD_stop}")
        self.send(f"YA 'ID', 1, -{ID_compliance}, {ID_compliance}")

        self.send("MD")   # measurement control page
        self.send("ME1")  # run measurement

    def run_IVG(self, VD_start, VD_step, VD_step_num, VG_start, VG_stop, VG_step, 
                  ID_compliance = 10e-6, IG_compliance = 10e-6, HT = 0, DT = 0, IT = 2, 
                  RS = 5, ID_lowrange = 1e-9, IG_lowrange = 1e-9):
        """ID–VG measurement, sweeping Vg with fixed Vd."""
        self.send("BC")
        self.send("DE")
        self.send("CH1, 'VD', 'ID', 1, 2")
        self.send("CH2, 'VG', 'IG', 1, 1")

        self.send("SS")
        self.send(f"VP1, {VD_start}, {VD_step}, {VD_step_num}, {ID_compliance}")          
        self.send(f"VR2, {VG_start}, {VG_stop}, {VG_step}, {IG_compliance}") 
        self.send(f"HT {HT}")
        self.send(f"DT {DT}")
        self.send(f"IT{IT}")
        self.send(f"RS {RS}")
        self.send(f"RG 1, {ID_lowrange}")
        self.send(f"RG 2, {IG_lowrange}")

        self.send("SM")
        self.send("DM1")
        self.send(f"XN 'VG', 2, {VG_start}, {VG_stop}")
        self.send(f"YA 'ID', 1, -{ID_compliance}, {ID_compliance}")

        self.send("MD")
        self.send("ME1")

In [4]:
# 4200 test
INST_RESOURCE_STR = "TCPIP0::169.254.16.30::1225::SOCKET"
my4200 = Keithley4200(INST_RESOURCE_STR)
my4200.connect()

#my4200.run_IVG(VG_start = -5, VG_stop = 5, VG_step = 0.1, VD_start = -5, VD_step = 2.5, VD_step_num = 5)
#my4200.wait_for_completion()
#my4200.save_data(filename="IVG_test")

In [5]:
my4200.run_IVD(VD_start = -5, VD_stop = 5, VD_step = 0.1, VG_start = -5, VG_step = 2.5, VG_step_num = 5)
my4200.wait_for_completion()
my4200.save_data(filename="IVD_test")

ValueError: invalid literal for int() with base 10: 'ACK'

In [7]:
print(my4200.send("SP", expect_response=True))

ACK


In [9]:
my4200.query("ME4")

AttributeError: 'Keithley4200' object has no attribute 'query'